In [1]:
import math
import inspect
from collections import deque

import numpy as np

np.set_printoptions(precision=4, suppress=True)

# Task 1

## Task 1.1

Datos medidos: $N = 1\,200$, $M = 3\,850$, $\langle C \rangle = 0.48$, $\langle d \rangle = 4.2$.

### a. Grado promedio $\langle k \rangle$

In [2]:
N, M = 1200, 3850
C_medido, d_medido = 0.48, 4.2

k_prom = 2 * M / N
print(f"<k> = 2M/N = 2({M})/{N} = {2*M}/{N} = {k_prom:.4f}")

<k> = 2M/N = 2(3850)/1200 = 7700/1200 = 6.4167


### b. Valores esperados en una red Erdős-Rényi equivalente

In [3]:
p_ER = k_prom / (N - 1)
C_aleatoria = k_prom / N
d_aleatoria = math.log(N) / math.log(k_prom)

print(f"p = <k>/(N-1) = {p_ER:.6f}")
print(f"C_aleatoria = <k>/N     = {C_aleatoria:.6f}")
print(f"ln N = {math.log(N):.4f}")
print(f"ln <k> = {math.log(k_prom):.4f}")
print(f"<d>_aleatoria = ln N / ln <k> = {d_aleatoria:.4f}")

p = <k>/(N-1) = 0.005352
C_aleatoria = <k>/N     = 0.005347
ln N = 7.0901
ln <k> = 1.8589
<d>_aleatoria = ln N / ln <k> = 3.8141


### c. Comparación y clasificación topológica

In [4]:
razon_C = C_medido / C_aleatoria
razon_d = d_medido / d_aleatoria
sigma = razon_C / razon_d   # coeficiente small-world (Humphries & Gurney, 2008)

print(f"{'Métrica':<10}{'Medida':>10}{'ER equivalente':>16}{'Razón medida/ER':>18}")
print(f"{'<C>':<10}{C_medido:>10.4f}{C_aleatoria:>16.4f}{razon_C:>18.2f}")
print(f"{'<d>':<10}{d_medido:>10.4f}{d_aleatoria:>16.4f}{razon_d:>18.2f}")
print(f"\nsigma = (C/C_ER) / (d/d_ER) = {sigma:.1f}")

Métrica       Medida  ER equivalente   Razón medida/ER
<C>           0.4800          0.0053             89.77
<d>           4.2000          3.8141              1.10

sigma = (C/C_ER) / (d/d_ER) = 81.5


## Task 1.2

In [5]:
A = np.array([
    [0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0],
    [1, 1, 0, 0, 1],
    [0, 1, 0, 0, 1],
    [0, 0, 1, 1, 0],
])
aristas = [(i + 1, j + 1) for i in range(len(A)) for j in range(i + 1, len(A)) if A[i, j]]
print("Simétrica:", np.array_equal(A, A.T), "| Diagonal nula:", not A.diagonal().any())
print("Aristas:", aristas, "| M =", len(aristas))

Simétrica: True | Diagonal nula: True
Aristas: [(1, 2), (1, 3), (2, 3), (2, 4), (3, 5), (4, 5)] | M = 6


## Task 1.3

In [6]:
def _validar_adyacencia(A):
    # Convierte a arreglo NumPy y verifica que sea una matriz de adyacencia no dirigida válida.
    A = np.asarray(A)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A debe ser una matriz cuadrada")
    if not np.array_equal(A, A.T):
        raise ValueError("A debe ser simétrica (red no dirigida)")
    if A.diagonal().any():
        raise ValueError("A no debe tener lazos (diagonal nula)")
    return A


def grado(A):
    # Retorna el grado k_i de cada nodo: la suma de la fila i de A.
    A = _validar_adyacencia(A)
    return A.sum(axis=1).astype(int)


def clustering(A):
    # Retorna el coeficiente de clustering C_i = e_i / (k_i (k_i - 1) / 2) de cada nodo.
    A = _validar_adyacencia(A)
    n = A.shape[0]
    C = np.zeros(n)
    for i in range(n):
        vecinos = np.flatnonzero(A[i])
        k_i = len(vecinos)
        if k_i < 2:
            continue  # C_i no está definido; por convención se deja en 0
        e_i = A[np.ix_(vecinos, vecinos)].sum() / 2  # aristas entre los vecinos de i
        C[i] = e_i / (k_i * (k_i - 1) / 2)
    return C


def bfs(A, origen):
    # Distancias geodésicas desde `origen` a todos los nodos; -1 si no hay camino.
    A = np.asarray(A)
    dist = np.full(A.shape[0], -1, dtype=int)
    dist[origen] = 0
    cola = deque([origen])
    while cola:
        u = cola.popleft()
        for v in np.flatnonzero(A[u]):
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                cola.append(v)
    return dist


def matriz_distancias(A):
    # Matriz D con D[i, j] = d_ij, obtenida con un BFS desde cada nodo.
    A = _validar_adyacencia(A)
    return np.array([bfs(A, i) for i in range(A.shape[0])])


def distancia_promedio(A):
    # Promedio de d_ij sobre los pares i < j que tienen camino entre sí.
    D = matriz_distancias(A)
    d_pares = D[np.triu_indices(D.shape[0], k=1)]
    d_pares = d_pares[d_pares > 0]  # descarta pares sin camino (-1)
    if d_pares.size == 0:
        return float("nan")
    return float(d_pares.mean())

### Verificación con la matriz del Task 1.2

In [7]:
k = grado(A)
C = clustering(A)
D = matriz_distancias(A)
d_prom = distancia_promedio(A)

print(f"{'Nodo':<6}{'k_i':>5}{'C_i':>10}")
for i in range(len(A)):
    print(f"{i+1:<6}{k[i]:>5}{C[i]:>10.4f}")
print(f"\n<k> = {k.mean():.4f}")
print(f"<C> = {C.mean():.4f}")

print("\nDistancias d_ij (triángulo superior):")
n = len(A)
print("     " + "".join(f"{j+1:>4}" for j in range(1, n)))
for i in range(n - 1):
    fila = "".join("    " if j <= i else f"{D[i, j]:>4}" for j in range(1, n))
    print(f"{i+1:>4} {fila}")
print(f"\n<d> = {d_prom:.4f}")

Nodo    k_i       C_i
1         2    1.0000
2         3    0.3333
3         3    0.3333
4         2    0.0000
5         2    0.0000

<k> = 2.4000
<C> = 0.3333

Distancias d_ij (triángulo superior):
        2   3   4   5
   1    1   1   2   2
   2        1   1   2
   3            2   1
   4                1

<d> = 1.4000


In [8]:
# Comparación automática con los resultados manuales del Task 1.2
k_manual = np.array([2, 3, 3, 2, 2])
C_manual = np.array([1, 1/3, 1/3, 0, 0])
d_manual = 14 / 10

assert np.array_equal(k, k_manual)
assert np.allclose(C, C_manual)
assert math.isclose(d_prom, d_manual)

# Verificación cruzada del clustering: (A^3)_ii = 2 * (número de triángulos que contienen a i)
triangulos = np.diag(np.linalg.matrix_power(A, 3)) / 2
C_alt = np.divide(triangulos, k * (k - 1) / 2, out=np.zeros(len(A)), where=k >= 2)
assert np.allclose(C, C_alt)

print("✔ grado, clustering y distancia_promedio coinciden con los cálculos manuales")
print("✔ clustering coincide con el cálculo alternativo por traza de A^3")

✔ grado, clustering y distancia_promedio coinciden con los cálculos manuales
✔ clustering coincide con el cálculo alternativo por traza de A^3


### Verificación de la predicción del Task 1.2d (nodo 6 conectado a 2 y 4)

In [9]:
A6 = np.zeros((6, 6), dtype=int)
A6[:5, :5] = A
for v in (2, 4):                  # nodos 2 y 4 (índices 1 y 3)
    A6[5, v - 1] = A6[v - 1, 5] = 1

k6, C6 = grado(A6), clustering(A6)
print(f"{'Nodo':<6}{'k_i antes':>10}{'k_i después':>13}{'C_i antes':>11}{'C_i después':>13}")
for i in range(6):
    ka = f"{k[i]}" if i < 5 else "—"
    Ca = f"{C[i]:.4f}" if i < 5 else "—"
    print(f"{i+1:<6}{ka:>10}{k6[i]:>13}{Ca:>11}{C6[i]:>13.4f}")
print(f"\n<k>: {k.mean():.4f} -> {k6.mean():.4f}")
print(f"<C>: {C.mean():.4f} -> {C6.mean():.4f}")
print(f"<d>: {d_prom:.4f} -> {distancia_promedio(A6):.4f}")

Nodo   k_i antes  k_i después  C_i antes  C_i después
1              2            2     1.0000       1.0000
2              3            4     0.3333       0.3333
3              3            3     0.3333       0.3333
4              2            3     0.0000       0.3333
5              2            2     0.0000       0.0000
6              —            2          —       1.0000

<k>: 2.4000 -> 2.6667
<C>: 0.3333 -> 0.5000
<d>: 1.4000 -> 1.4667


Se confirma el argumento del inciso d: $\langle k\rangle$ pasa de 2.4 a 2.667 y $\langle C\rangle$ **sube** de 0.333 a 0.5. El aumento viene del nodo 6 ($C_6 = 1$) y del nodo 4 (de 0 a 1/3). El nodo 2 se mantiene en 1/3 porque tanto el numerador como el denominador crecieron.